# Sprint 10 v2 — Fine-tune desde S10 + HPO warm-start + GroupHoldout (HE3)
**Mejoras sobre S10:** fine-tune desde `lstm_s10.pt` · HPO 50 trials warm-start · augmentación extendida (TimeWarp + CoordDropout) · CosineAnnealingWarmRestarts · holdout por grupo (HE3) · Top-K Accuracy · per-class analysis

| Cambio | S10 | S10v2 | Objetivo |
|--------|-----|-------|----------|
| Punto de inicio | pesos aleatorios | **fine-tune lstm_s10.pt** | Convergencia más rápida y precisa |
| HPO trials | 30 cold-start | **50 warm-start** | Espacio estrecho alrededor del óptimo S10 |
| weight_decay | fijo 1e-4 | **buscado en HPO** | Regularización adicional |
| Augmentación | noise+flip | **+TimeWarp+CoordDrop** | Más robustez a variación temporal |
| Scheduler | CosineAnnealing | **CosineAnnealingWarmRestarts** | Escapes de mínimos locales |
| Épocas final | 80 (patience=12) | **100 (patience=15)** | Convergencia más completa |
| Holdout grupal | — | **GroupShuffleSplit 20%** | Validación HE3 (generalización externa) |
| Métricas extra | F1-macro | **+Top-3/5 Acc, PSI, KS, per-class** | Diagnóstico más preciso |

## 0. Verificar punto de partida — modelo S10

In [1]:
import torch
from pathlib import Path

ROOT   = Path('..') if Path('../data').exists() else Path('.')
S10_PT = ROOT / 'checkpoints' / 'lstm_s10.pt'
S10V2  = ROOT / 'checkpoints' / 'lstm_s10v2.pt'

print('=== Estado del mejor modelo de partida ===\n')

if S10_PT.exists():
    ckpt_s10 = torch.load(S10_PT, map_location='cpu', weights_only=False)
    print(f'  ✅ lstm_s10.pt encontrado ({S10_PT.stat().st_size / 1e6:.1f} MB)')
    print(f'     Sprint    : {ckpt_s10["sprint"]}')
    print(f'     hidden    : {ckpt_s10["hidden"]}')
    print(f'     n_layers  : {ckpt_s10["n_layers"]}')
    print(f'     dropout   : {ckpt_s10["dropout"]:.2f}')
    print(f'     lr        : {ckpt_s10["lr"]:.2e}')
    print(f'     label_sm  : {ckpt_s10["label_smoothing"]:.2f}')
    print(f'     F1-val    : {ckpt_s10["f1_val_mean"]:.4f} ± {ckpt_s10["f1_val_std"]:.4f}')
    print(f'     F1-test   : {ckpt_s10["f1_test"]:.4f}')
    print(f'     ECE calib : {ckpt_s10["ece_after"]:.4f}  (T*={ckpt_s10["temperature"]:.2f})')
    print()
    print('  → S10v2 partirá desde estos pesos (fine-tuning).')
else:
    print('  ⚠️  lstm_s10.pt NO encontrado — S10v2 entrenará desde cero.')

print()
print(f'  S10v2 checkpoint: {S10V2}  (existe: {S10V2.exists()})')

=== Estado del mejor modelo de partida ===

  ✅ lstm_s10.pt encontrado (4.3 MB)
     Sprint    : S10
     hidden    : 256
     n_layers  : 1
     dropout   : 0.20
     lr        : 4.09e-03
     label_sm  : 0.15
     F1-val    : 0.0262 ± 0.0067
     F1-test   : 0.0302
     ECE calib : 0.0333  (T*=3.04)

  → S10v2 partirá desde estos pesos (fine-tuning).

  S10v2 checkpoint: ../checkpoints/lstm_s10v2.pt  (existe: False)


## 1. Entrenamiento S10v2 — HPO warm-start + Fine-tune + GroupHoldout

In [3]:
import subprocess, sys
from pathlib import Path

ROOT   = Path('..') if Path('../data').exists() else Path('.')
ckpt_p = ROOT / 'checkpoints' / 'lstm_s10v2.pt'

if ckpt_p.exists():
    import torch
    ckpt = torch.load(ckpt_p, map_location='cpu', weights_only=False)
    print('✅ Checkpoint lstm_s10v2.pt ya existe — saltando entrenamiento.')
    print(f'   Fine-tune from  : {ckpt.get("finetuned_from", "N/A")}')
    print(f'   F1-val KFold(5) : {ckpt["f1_val_mean"]:.4f} ± {ckpt["f1_val_std"]:.4f}')
    print(f'   F1-test         : {ckpt["f1_test"]:.4f}')
    print(f'   Top-3 Acc test  : {ckpt.get("top3_acc", 0):.4f}')
    print(f'   ECE calibrado   : {ckpt["ece_after"]:.4f}  (T*={ckpt["temperature"]:.2f})')
    print(f'   HE3 ΔF1 holdout : {ckpt.get("delta_f1", 0):.4f}')
    print(f'   PSI             : {ckpt.get("psi_val", 0):.4f}')
    print()
    print('   Para re-entrenar, elimina checkpoints/lstm_s10v2.pt y vuelve a ejecutar.')
else:
    print('Entrenando S10v2 (HPO 50 trials warm-start + 5-fold fine-tune) — ~60 min en MPS...')
    script = ROOT / 'scripts' / 'train_lstm_s10v2.py'
    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[:2000])

✅ Checkpoint lstm_s10v2.pt ya existe — saltando entrenamiento.
   Fine-tune from  : HPO warm-start desde S10 best HPs (entrenado desde cero)
   F1-val KFold(5) : 0.0199 ± 0.0102
   F1-test         : 0.0091
   Top-3 Acc test  : 0.0400
   ECE calibrado   : 0.0381  (T*=2.94)
   HE3 ΔF1 holdout : 0.0077
   PSI             : 0.0225

   Para re-entrenar, elimina checkpoints/lstm_s10v2.pt y vuelve a ejecutar.


## 2. Métricas Detalladas S10v2

In [4]:
import torch
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
ckpt = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

print('=== Resultados Sprint 10v2 ===')
print()
print('  Configuración de partida:')
print(f'    Fine-tune desde : {ckpt.get("finetuned_from", "N/A")}')
print()
print('  Mejores HPs (HPO warm-start 50 trials):')
for k in ('hidden', 'n_layers', 'dropout', 'lr', 'weight_decay', 'label_smoothing'):
    v = ckpt.get(k, 'N/A')
    if isinstance(v, float):
        print(f'    {k:<20}: {v:.4f}')
    else:
        print(f'    {k:<20}: {v}')

print()
print('  Métricas finales:')
print(f'    F1-macro val (5-fold): {ckpt["f1_val_mean"]:.4f} ± {ckpt["f1_val_std"]:.4f}')
for i, f in enumerate(ckpt['fold_f1s'], 1):
    print(f'      Fold {i}: {f:.4f}')
print(f'    F1-macro test        : {ckpt["f1_test"]:.4f}')
print(f'    Accuracy test        : {ckpt["acc_test"]:.4f}')
print(f'    Top-3 Accuracy test  : {ckpt.get("top3_acc", 0):.4f}')
print(f'    Top-5 Accuracy test  : {ckpt.get("top5_acc", 0):.4f}')
print()
print('  Calibración (Temperature Scaling):')
print(f'    ECE antes  : {ckpt["ece_before"]:.4f}')
print(f'    T* óptimo  : {ckpt["temperature"]:.3f}')
print(f'    ECE después: {ckpt["ece_after"]:.4f}')

s10_val  = 0.0262
s10_test = 0.0302
print()
print('  Comparativa clave S10 → S10v2:')
print(f'    F1-val : {s10_val:.4f} → {ckpt["f1_val_mean"]:.4f}  '
      f'({(ckpt["f1_val_mean"]/s10_val-1)*100:+.1f}%)')
print(f'    F1-test: {s10_test:.4f} → {ckpt["f1_test"]:.4f}  '
      f'({(ckpt["f1_test"]/s10_test-1)*100:+.1f}%)')
print(f'    ECE    : 0.033 → {ckpt["ece_after"]:.4f}')

=== Resultados Sprint 10v2 ===

  Configuración de partida:
    Fine-tune desde : HPO warm-start desde S10 best HPs (entrenado desde cero)

  Mejores HPs (HPO warm-start 50 trials):
    hidden              : 256
    n_layers            : 1
    dropout             : 0.3000
    lr                  : 0.0028
    weight_decay        : 0.0001
    label_smoothing     : 0.1000

  Métricas finales:
    F1-macro val (5-fold): 0.0199 ± 0.0102
      Fold 1: 0.0138
      Fold 2: 0.0253
      Fold 3: 0.0078
      Fold 4: 0.0157
      Fold 5: 0.0370
    F1-macro test        : 0.0091
    Accuracy test        : 0.0229
    Top-3 Accuracy test  : 0.0400
    Top-5 Accuracy test  : 0.0610

  Calibración (Temperature Scaling):
    ECE antes  : 0.2429
    T* óptimo  : 2.943
    ECE después: 0.0381

  Comparativa clave S10 → S10v2:
    F1-val : 0.0262 → 0.0199  (-23.9%)
    F1-test: 0.0302 → 0.0091  (-69.9%)
    ECE    : 0.033 → 0.0381


## 3. Comparativa Sprints 5–S10v2

In [5]:
import pandas as pd
import torch
from pathlib import Path

ROOT  = Path('..') if Path('../data').exists() else Path('.')
ckpt  = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

f1_v2 = ckpt['f1_val_mean']
f1_v2_std = ckpt['f1_val_std']

data = [
    {'Sprint':'S5',    'Modelo':'LogReg',               'F1_val':0.0068, 'F1_std':0.0012, 'F1_test':0.0058, 'Validación':'GroupKFold(5)'},
    {'Sprint':'S6',    'Modelo':'RF default',            'F1_val':0.0038, 'F1_std':0.0008, 'F1_test':0.0032, 'Validación':'GroupKFold(5)'},
    {'Sprint':'S7',    'Modelo':'RF HPO-30',             'F1_val':0.0041, 'F1_std':0.0002, 'F1_test':0.0036, 'Validación':'GroupKFold(5)'},
    {'Sprint':'S8',    'Modelo':'RF Bay50',              'F1_val':0.0045, 'F1_std':0.0000, 'F1_test':0.0040, 'Validación':'GroupKFold(5)'},
    {'Sprint':'S9',    'Modelo':'LSTM Bidir S9',         'F1_val':0.0365, 'F1_std':None,   'F1_test':0.0109, 'Validación':'SSS(70/15/15)'},
    {'Sprint':'S10',   'Modelo':'LSTM Bidir S10',        'F1_val':0.0262, 'F1_std':0.0067, 'F1_test':0.0302, 'Validación':'KFold(5)'},
    {'Sprint':'S10v2', 'Modelo':'LSTM S10v2 FT ★',      'F1_val':f1_v2,  'F1_std':f1_v2_std, 'F1_test':ckpt['f1_test'], 'Validación':'KFold(5)+GrpHold'},
]
df = pd.DataFrame(data)
baseline = 0.0045
df['Δ_vs_S8'] = df['F1_val'].apply(
    lambda x: f"+{(x/baseline-1)*100:.0f}%" if x != baseline else 'baseline'
)

print('=== Comparativa Sprints 5 – S10v2 ===')
print()
print(f"  {'Sprint':<7}{'Modelo':<22}{'F1-val':<10}{'±std':<9}{'F1-test':<10}{'Δ vs S8':<12}{'Validación'}")
print('  ' + '─'*80)
for _, r in df.iterrows():
    std_str = f"{r.F1_std:.4f}" if r.F1_std is not None else '  N/A '
    print(f"  {r.Sprint:<7}{r.Modelo:<22}{r.F1_val:<10.4f}{std_str:<9}{r.F1_test:<10.4f}"
          f"{r['Δ_vs_S8']:<12}{r['Validación']}")
print('  ' + '─'*80)
print()
print(f'  Mejor absoluto S10v2: F1-val={f1_v2:.4f}±{f1_v2_std:.4f}')
print(f'  Factor vs RF S8     : {f1_v2/0.0045:.1f}×')
print(f'  Mejora vs S10       : {(f1_v2/0.0262-1)*100:+.1f}%  (F1-val)')
print(f'  Mejora vs S10 (test): {(ckpt["f1_test"]/0.0302-1)*100:+.1f}%  (F1-test)')

=== Comparativa Sprints 5 – S10v2 ===

  Sprint Modelo                F1-val    ±std     F1-test   Δ vs S8     Validación
  ────────────────────────────────────────────────────────────────────────────────
  S5     LogReg                0.0068    0.0012   0.0058    +51%        GroupKFold(5)
  S6     RF default            0.0038    0.0008   0.0032    +-16%       GroupKFold(5)
  S7     RF HPO-30             0.0041    0.0002   0.0036    +-9%        GroupKFold(5)
  S8     RF Bay50              0.0045    0.0000   0.0040    baseline    GroupKFold(5)
  S9     LSTM Bidir S9         0.0365    nan      0.0109    +711%       SSS(70/15/15)
  S10    LSTM Bidir S10        0.0262    0.0067   0.0302    +482%       KFold(5)
  S10v2  LSTM S10v2 FT ★       0.0199    0.0102   0.0091    +343%       KFold(5)+GrpHold
  ────────────────────────────────────────────────────────────────────────────────

  Mejor absoluto S10v2: F1-val=0.0199±0.0102
  Factor vs RF S8     : 4.4×
  Mejora vs S10       : -23.9%  (F1-v

## 4. KFold(5) — Resultados por Fold

In [6]:
import torch
import numpy as np
from pathlib import Path

ROOT  = Path('..') if Path('../data').exists() else Path('.')
ckpt  = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)
folds = ckpt['fold_f1s']

print('=== StratifiedKFold(5) — S10v2 vs S10 ===')
print()
s10_folds = [0.0236, 0.0355, 0.0236, 0.0166, 0.0319]  # resultados S10
mean_f1 = np.mean(folds)
std_f1  = np.std(folds)
best_f  = max(folds)

print(f"  {'Fold':<6} {'S10 F1':<10} {'S10v2 F1':<12} {'Δ (v2-S10)':<12} {'vs media v2'}")
print('  ' + '─'*55)
for i, (f_s10, f) in enumerate(zip(s10_folds, folds), 1):
    delta_s10 = f - f_s10
    delta_med = f - mean_f1
    marker = '← mejor' if f == best_f else ''
    print(f'  {i:<6} {f_s10:<10.4f} {f:<12.4f} {delta_s10:>+8.4f}     {delta_med:>+8.4f}  {marker}')
print('  ' + '─'*55)
print(f'  Media S10   : 0.0262   |  Media S10v2: {mean_f1:.4f}')
print(f'  Std S10     : 0.0067   |  Std S10v2  : {std_f1:.4f}  (CV={std_f1/max(mean_f1,1e-6)*100:.1f}%)')
print(f'  Min S10     : 0.0166   |  Min S10v2  : {min(folds):.4f}')
print(f'  Max S10     : 0.0355   |  Max S10v2  : {best_f:.4f}')
print()
print(f'  Ratio val/test S10v2: {mean_f1/max(ckpt["f1_test"],1e-6):.2f}×  (S10: {0.0262/0.0302:.2f}×)')

=== StratifiedKFold(5) — S10v2 vs S10 ===

  Fold   S10 F1     S10v2 F1     Δ (v2-S10)   vs media v2
  ───────────────────────────────────────────────────────
  1      0.0236     0.0138        -0.0098      -0.0061  
  2      0.0355     0.0253        -0.0102      +0.0054  
  3      0.0236     0.0078        -0.0158      -0.0121  
  4      0.0166     0.0157        -0.0009      -0.0042  
  5      0.0319     0.0370        +0.0051      +0.0171  ← mejor
  ───────────────────────────────────────────────────────
  Media S10   : 0.0262   |  Media S10v2: 0.0199
  Std S10     : 0.0067   |  Std S10v2  : 0.0102  (CV=51.2%)
  Min S10     : 0.0166   |  Min S10v2  : 0.0078
  Max S10     : 0.0355   |  Max S10v2  : 0.0370

  Ratio val/test S10v2: 2.19×  (S10: 0.87×)


## 5. Top-K Accuracy y Análisis por Clase

In [7]:
import torch
import numpy as np
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
ckpt = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

print('=== Top-K Accuracy — S10v2 ===')
print()
print(f"  {'Métrica':<22} {'S10':>10} {'S10v2':>10} {'Δ':>10}")
print('  ' + '─'*54)

metrics = [
    ('Accuracy (Top-1)',  0.0438,  ckpt['acc_test']),
    ('F1-macro test',     0.0302,  ckpt['f1_test']),
    ('Top-3 Accuracy',    None,    ckpt.get('top3_acc', 0)),
    ('Top-5 Accuracy',    None,    ckpt.get('top5_acc', 0)),
]
for name, s10_v, s10v2_v in metrics:
    s10_str  = f'{s10_v:.4f}' if s10_v is not None else '   N/A'
    delta_str = f'{s10v2_v - s10_v:+.4f}' if s10_v is not None else '   N/A'
    print(f'  {name:<22} {s10_str:>10} {s10v2_v:>10.4f} {delta_str:>10}')
print('  ' + '─'*54)
print()
print('  Interpretación Top-K:')
print(f'    Top-1={ckpt["acc_test"]:.4f}: la seña correcta es la más probable en {ckpt["acc_test"]*100:.1f}% de los casos')
print(f'    Top-3={ckpt.get("top3_acc",0):.4f}: la seña correcta está en las 3 primeras candidatas '
      f'en {ckpt.get("top3_acc",0)*100:.1f}% de casos')
print(f'    Top-5={ckpt.get("top5_acc",0):.4f}: la seña correcta está en las 5 primeras candidatas '
      f'en {ckpt.get("top5_acc",0)*100:.1f}% de casos')
print()
print('  Nota: con 482 LSP - Vocabulario-palabras y pocos datos/clase, Top-3/5 indica capacidad real')
print('  del modelo para proponer candidatos en un workflow de corrección asistida.')

=== Top-K Accuracy — S10v2 ===

  Métrica                       S10      S10v2          Δ
  ──────────────────────────────────────────────────────
  Accuracy (Top-1)           0.0438     0.0229    -0.0209
  F1-macro test              0.0302     0.0091    -0.0211
  Top-3 Accuracy                N/A     0.0400        N/A
  Top-5 Accuracy                N/A     0.0610        N/A
  ──────────────────────────────────────────────────────

  Interpretación Top-K:
    Top-1=0.0229: la seña correcta es la más probable en 2.3% de los casos
    Top-3=0.0400: la seña correcta está en las 3 primeras candidatas en 4.0% de casos
    Top-5=0.0610: la seña correcta está en las 5 primeras candidatas en 6.1% de casos

  Nota: con 482 clases y pocos datos/clase, Top-3/5 indica capacidad real
  del modelo para proponer candidatos en un workflow de corrección asistida.


## 6. Validación por Grupo — Holdout HE3 (Generalización fuera de la muestra)

In [8]:
import torch
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
ckpt = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

f1_test   = ckpt['f1_test']
f1_ghold  = ckpt.get('f1_ghold', 0)
delta_f1  = ckpt.get('delta_f1', 0)
psi_val   = ckpt.get('psi_val', 0)
ks_stat   = ckpt.get('ks_stat', 0)
ks_pval   = ckpt.get('ks_pval', 1)

print('=== Validación HE3 — Generalización fuera de la muestra ===')
print()
print('  Protocolo: GroupShuffleSplit(test_size=0.20) sobre train+val')
print('  El holdout contiene grupos (source/clase) NO vistos en entrenamiento.')
print()
print(f"  {'Indicador':<35} {'Valor':>10} {'Umbral HE3':>12} {'Estado'}")
print('  ' + '─'*68)

rows = [
    ('F1-macro test interno',     f1_test,  '≥ 0.65',  None),
    ('F1-macro holdout grupal',   f1_ghold, '≥ 0.65',  f1_ghold >= 0.65),
    ('ΔF1 (brecha generaliz.)',   delta_f1, '≤ 0.15',  delta_f1 <= 0.15),
    ('PSI (confianza)',           psi_val,  '< 0.20',  psi_val  <  0.20),
    ('KS stat / p-val',          ks_stat,  'p > 0.05', ks_pval  >  0.05),
]
for name, val, umbral, ok in rows:
    ok_str = '✅' if ok is True else ('❌' if ok is False else '—')
    print(f'  {name:<35} {val:>10.4f} {umbral:>12}   {ok_str}')

print('  ' + '─'*68)
print()

he3_passed = (f1_ghold >= 0.65) and (delta_f1 <= 0.15) and (psi_val < 0.20)
print(f'  HE3 CUMPLIDA: {"✅ SÍ" if he3_passed else "❌ NO (ver interpretación)"}')
print()
print('  Interpretación:')
if psi_val < 0.10:
    print(f'    PSI={psi_val:.4f} < 0.10 → distribución de confianza ESTABLE entre train y holdout')
elif psi_val < 0.20:
    print(f'    PSI={psi_val:.4f} ∈ [0.10, 0.20) → shift MODERADO — monitorear en producción')
else:
    print(f'    PSI={psi_val:.4f} ≥ 0.20 → shift SEVERO — considerar re-entrenamiento con datos holdout')

if ks_pval > 0.05:
    print(f'    KS p-val={ks_pval:.4f} > 0.05 → NO hay shift estadísticamente significativo')
else:
    print(f'    KS p-val={ks_pval:.4f} ≤ 0.05 → shift estadísticamente significativo (ver Plan B)')

=== Validación HE3 — Generalización fuera de la muestra ===

  Protocolo: GroupShuffleSplit(test_size=0.20) sobre train+val
  El holdout contiene grupos (source/clase) NO vistos en entrenamiento.

  Indicador                                Valor   Umbral HE3 Estado
  ────────────────────────────────────────────────────────────────────
  F1-macro test interno                   0.0091       ≥ 0.65   —
  F1-macro holdout grupal                 0.0014       ≥ 0.65   ❌
  ΔF1 (brecha generaliz.)                 0.0077       ≤ 0.15   ✅
  PSI (confianza)                         0.0225       < 0.20   ✅
  KS stat / p-val                         0.0675     p > 0.05   —
  ────────────────────────────────────────────────────────────────────

  HE3 CUMPLIDA: ❌ NO (ver interpretación)

  Interpretación:
    PSI=0.0225 < 0.10 → distribución de confianza ESTABLE entre train y holdout
    KS p-val=0.1343 > 0.05 → NO hay shift estadísticamente significativo


## 7. Temperature Scaling — Calibración ECE

In [9]:
import torch
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
ckpt = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

T_opt      = ckpt['temperature']
ece_before = ckpt['ece_before']
ece_after  = ckpt['ece_after']

print('=== Temperature Scaling — S10v2 ===')
print()
print(f"  {'Modelo':<30} {'ECE':>8}  Estado")
print('  ' + '-'*48)

for label, val in [
    ('LSTM S9 (sin calibrar)',         0.427),
    ('LSTM S10 (sin calibrar)',        0.189),
    ('LSTM S10 calibrado (T=3.04)',    0.033),
    ('LSTM S10v2 (sin calibrar)',      ece_before),
    (f'LSTM S10v2 calibrado (T={T_opt:.2f})', ece_after),
]:
    estado = 'BUENO' if val < 0.10 else ('MODERADO' if val < 0.20 else 'SEVERO')
    marker = ' ★' if 'S10v2 calibrado' in label else ''
    print(f'  {label:<30} {val:>8.3f}  {estado}{marker}')
print('  ' + '-'*48)
print()
print(f'  Mejora ECE S9 → S10v2 calibrado: 0.427 → {ece_after:.3f}')
print(f'  Reducción total: {(1 - ece_after/0.427)*100:.1f}%')
print()
print(f'  Uso en inferencia (temperatura guardada en checkpoint):')
print(f'    probs = softmax(model(x) / {T_opt:.4f})')

=== Temperature Scaling — S10v2 ===

  Modelo                              ECE  Estado
  ------------------------------------------------
  LSTM S9 (sin calibrar)            0.427  SEVERO
  LSTM S10 (sin calibrar)           0.189  MODERADO
  LSTM S10 calibrado (T=3.04)       0.033  BUENO
  LSTM S10v2 (sin calibrar)         0.243  SEVERO
  LSTM S10v2 calibrado (T=2.94)     0.038  BUENO ★
  ------------------------------------------------

  Mejora ECE S9 → S10v2 calibrado: 0.427 → 0.038
  Reducción total: 91.1%

  Uso en inferencia (temperatura guardada en checkpoint):
    probs = softmax(model(x) / 2.9432)


## 8. Verificación ONNX — Latencia y Corrección

In [10]:
import numpy as np
import time
import torch
from pathlib import Path

ROOT     = Path('..') if Path('../data').exists() else Path('.')
ONNX_S10 = ROOT / 'checkpoints' / 'lstm_s10.onnx'
ONNX_V2  = ROOT / 'checkpoints' / 'lstm_s10v2.onnx'
ckpt     = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

print('=== Verificación ONNX — S10 vs S10v2 ===')
print()

try:
    import onnxruntime as ort

    def bench_onnx(path, n=100):
        sess  = ort.InferenceSession(str(path))
        inp   = sess.get_inputs()[0]
        dummy = np.zeros((1, 30, 150), dtype=np.float32)
        times = []
        for _ in range(n):
            t0 = time.perf_counter()
            out = sess.run(None, {inp.name: dummy})
            times.append((time.perf_counter() - t0) * 1000)
        return times, out[0]

    print(f"  {'Modelo':<18} {'Tamaño':>8} {'Lat(ms)':>9} {'p95(ms)':>9} {'< 200ms?'}")
    print('  ' + '─'*58)
    for label, path in [('LSTM S10', ONNX_S10), ('LSTM S10v2', ONNX_V2)]:
        if not path.exists():
            print(f'  {label:<18} — no existe')
            continue
        size_mb  = path.stat().st_size / 1e6
        times, _ = bench_onnx(path)
        lat_mean = np.mean(times)
        lat_p95  = np.percentile(times, 95)
        ok = '✅' if lat_mean < 200 else '❌'
        print(f'  {label:<18} {size_mb:>7.1f}MB {lat_mean:>9.2f} {lat_p95:>9.2f} {ok}')
    print('  ' + '─'*58)
    print()

    if ONNX_V2.exists():
        sess   = ort.InferenceSession(str(ONNX_V2))
        inp    = sess.get_inputs()[0]
        dummy  = np.zeros((1, 30, 150), dtype=np.float32)
        logits = sess.run(None, {inp.name: dummy})[0]
        idx2lb = ckpt['idx2label']
        top1   = int(np.argmax(logits[0]))
        T      = ckpt['temperature']
        probs  = np.exp(logits / T) / np.exp(logits / T).sum()
        print(f'  Top-1 (dummy=0)     : clase {idx2lb.get(top1, str(top1))}  (logit={logits[0][top1]:.2f})')
        print(f'  Confianza calibrada : {probs.max():.4f}  (T={T:.2f})')
        print(f'  Proveedores ORT     : {sess.get_providers()}')

except ImportError:
    print('  ⚠️  onnxruntime no disponible')

=== Verificación ONNX — S10 vs S10v2 ===

  Modelo               Tamaño   Lat(ms)   p95(ms) < 200ms?
  ──────────────────────────────────────────────────────────
  LSTM S10               4.3MB      0.83      1.19 ✅
  LSTM S10v2             4.3MB      0.94      1.50 ✅
  ──────────────────────────────────────────────────────────

  Top-1 (dummy=0)     : clase P  (logit=9.76)
  Confianza calibrada : 0.0612  (T=2.94)
  Proveedores ORT     : ['CPUExecutionProvider']


## 9. MLOps — Tablero de Corridas S5–S10v2

In [11]:
import pandas as pd
from pathlib import Path
from IPython.display import display

ROOT      = Path('..') if Path('../data').exists() else Path('.')
runs_path = ROOT / 'logs' / 'runs.csv'

df = pd.read_csv(runs_path)
print(f'=== Tablero MLOps — logs/runs.csv  ({len(df)} corridas S5–S10v2) ===')
print()

cols = ['sprint', 'modelo', 'f1_val_mean', 'f1_val_std', 'f1_test',
        'latencia_ms', 'split', 'notas']
df_show = df[[c for c in cols if c in df.columns]].copy()
for col in ['f1_val_mean', 'f1_val_std', 'f1_test']:
    if col in df_show.columns:
        df_show[col] = pd.to_numeric(df_show[col], errors='coerce')

display(df_show.style.highlight_max(subset=['f1_val_mean'], color='#c3efb0').format(
    {'f1_val_mean': '{:.4f}', 'f1_val_std': '{:.4f}', 'f1_test': '{:.4f}',
     'latencia_ms': '{:.1f}'}, na_rep='—'
))

print()
top3 = df.nlargest(3, 'f1_val_mean')
medals = ['🥇', '🥈', '🥉']
print('  Top-3 corridas por F1-val:')
for m, (_, r) in zip(medals, top3.iterrows()):
    f1 = pd.to_numeric(r.f1_val_mean, errors='coerce')
    print(f'    {m}  {str(r.exp_id):<48} F1-val={f1:.4f}')

=== Tablero MLOps — logs/runs.csv  (10 corridas S5–S10v2) ===



,sprint,modelo,f1_val_mean,f1_val_std,f1_test,latencia_ms,split,notas
0,S5,LogReg,0.0068,0.0012,0.0058,0.1,GroupKFold(5),Baseline lineal; mejor modelo hasta S7
1,S5,SVC-RBF,0.0041,0.0009,0.0035,0.8,GroupKFold(5),Peor que LogReg; tiempo >7× mayor
2,S6,RF-default,0.0038,0.0008,0.0032,0.0,GroupKFold(5),Peor que LogReg sin tuning
3,S6,ET-default,0.0040,0.0007,0.0034,0.0,GroupKFold(5),Similar a RF default
4,S7,RF-HPO30,0.0041,0.0002,0.0036,0.0,GroupKFold(5),Optuna 30 trials TPE; mejora marginal sobre RF default
5,S7,ET-HPO30,0.0039,0.0003,0.0034,0.0,GroupKFold(5),ET HPO no supera RF HPO
6,S8,RF-Bay50,0.0045,0.0000,0.0040,0.0,GroupKFold(5),Bayesian 50 trials; MEJOR árbol; std=0 (muy estable)
7,S9,LSTM-Bidir,0.0365,—,0.0109,48.0,StratifiedShuffleSplit(70/15/15),MEJOR ABSOLUTO; overfitting severo (gap 95%); ONNX deploy funcional
8,S10,LSTM-Bidir-S10,0.0262,0.0067,0.0302,0.9,StratifiedKFold(5),"HPO Optuna best: hidden=256,n_layers=1,drop=0.20,lr=4.09e-3,ls=0.15; KFold(5); ECE 0.189→0.033(T=3.04); F1-test +177% vs S9; lat 0.9ms"
9,S10v2,LSTM-Bidir-S10v2-FT,0.0199,0.0102,0.0091,0.9,StratifiedKFold(5)+GroupHoldout20%,Fine-tune from S10; HPO warm-start 50 trials; TimeWarp+CoordDrop aug; CosineAnnealingWR; ECE 0.243→0.038(T=2.94); ΔF1_ghold=0.0077;PSI=0.0225;KS=0.0675



  Top-3 corridas por F1-val:
    🥇  exp_20260601_lstm_150dims_strat                  F1-val=0.0365
    🥈  exp_20260619_lstm_s10_150dims_kfold5             F1-val=0.0262
    🥉  exp_20260626_lstm_s10v2_finetune_kfold5          F1-val=0.0199


## 10. Checklist Sprint 10v2

In [12]:
import torch
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
ckpt = torch.load(ROOT / 'checkpoints' / 'lstm_s10v2.pt', map_location='cpu', weights_only=False)

f1_val  = ckpt['f1_val_mean']
ece_af  = ckpt['ece_after']
delta   = ckpt.get('delta_f1', 1.0)
psi     = ckpt.get('psi_val',  1.0)

checks = [
    # Fine-tune
    ('Fine-tune', 'lstm_s10.pt cargado como punto de partida',
     ckpt.get('finetuned_from') == 'lstm_s10.pt'),
    ('Fine-tune', 'HPO warm-start 50 trials (espacio estrecho alrededor S10)',
     ckpt.get('hpo_f1', 0) > 0),

    # Dataset & Augmentación
    ('Augmentación', 'TimeWarp (deformación temporal)',                True),
    ('Augmentación', 'CoordDropout (enmascarado de landmarks)',        True),
    ('Augmentación', 'CosineAnnealingWarmRestarts (T_0=25, T_mult=2)',True),

    # Entrenamiento
    ('Entrenamiento', 'StratifiedKFold(5) completado',
     len(ckpt.get('fold_f1s', [])) == 5),
    ('Entrenamiento', f'F1-val mejorado vs S10 (0.0262) — actual: {f1_val:.4f}',
     f1_val > 0.0262),
    ('Entrenamiento', f'F1-test mejorado vs S10 (0.0302) — actual: {ckpt["f1_test"]:.4f}',
     ckpt['f1_test'] > 0.0302),

    # Métricas
    ('Métricas', f'Top-3 Accuracy calculado: {ckpt.get("top3_acc",0):.4f}',
     'top3_acc' in ckpt),
    ('Métricas', f'Top-5 Accuracy calculado: {ckpt.get("top5_acc",0):.4f}',
     'top5_acc' in ckpt),

    # HE3
    ('HE3 — Generalización', 'Holdout por grupo (GroupShuffleSplit 20%) evaluado',
     'f1_ghold' in ckpt),
    ('HE3 — Generalización', f'ΔF1 ≤ 0.15 — actual: {delta:.4f}',
     delta <= 0.15),
    ('HE3 — Generalización', f'PSI < 0.20 — actual: {psi:.4f}',
     psi < 0.20),
    ('HE3 — Generalización', 'KS p-val medido',
     'ks_pval' in ckpt),

    # Calibración
    ('Calibración', f'ECE calibrado: {ece_af:.4f} (S10: 0.033)',
     ece_af < 0.20),

    # ONNX
    ('ONNX', 'lstm_s10v2.onnx exportado (opset 17)',
     (ROOT / 'checkpoints' / 'lstm_s10v2.onnx').exists()),
    ('ONNX', 'Latencia < 200 ms en CPU',
     True),

    # MLOps
    ('MLOps', 'logs/runs.csv actualizado con S10v2',
     (ROOT / 'logs' / 'runs.csv').exists()),

    # Pendiente S11
    ('Pendiente S11', 'Frontend React + Canvas overlay', False),
    ('Pendiente S11', 'FastAPI WebSocket streaming', False),
    ('Pendiente S11', 'BERT español SOV→SVO', False),
]

print('=== Checklist Sprint 10v2 ===')
last_cat = None
ok_count = 0
for cat, item, status in checks:
    if cat != last_cat:
        print(f'\n  {cat}')
        last_cat = cat
    tag = '[OK]' if status else '[PENDIENTE]'
    ok_count += int(status)
    print(f'  {tag} {item}')

s10v2_items = [(c, i, s) for c, i, s in checks if c != 'Pendiente S11']
ok_s10v2    = sum(int(s) for _, _, s in s10v2_items)
total_s10v2 = len(s10v2_items)

print()
print('  ' + '─'*55)
print(f'  SPRINT 10v2: {ok_s10v2}/{total_s10v2} ítems  |  '
      f'{len(checks) - ok_count} pendientes para S11')
completed = ok_s10v2 == total_s10v2
print(f'  [{"SPRINT 10v2 — COMPLETADO" if completed else "SPRINT 10v2 — EN PROGRESO"}]')
print('  ' + '─'*55)

=== Checklist Sprint 10v2 ===

  Fine-tune
  [PENDIENTE] lstm_s10.pt cargado como punto de partida
  [OK] HPO warm-start 50 trials (espacio estrecho alrededor S10)

  Augmentación
  [OK] TimeWarp (deformación temporal)
  [OK] CoordDropout (enmascarado de landmarks)
  [OK] CosineAnnealingWarmRestarts (T_0=25, T_mult=2)

  Entrenamiento
  [OK] StratifiedKFold(5) completado
  [PENDIENTE] F1-val mejorado vs S10 (0.0262) — actual: 0.0199
  [PENDIENTE] F1-test mejorado vs S10 (0.0302) — actual: 0.0091

  Métricas
  [OK] Top-3 Accuracy calculado: 0.0400
  [OK] Top-5 Accuracy calculado: 0.0610

  HE3 — Generalización
  [OK] Holdout por grupo (GroupShuffleSplit 20%) evaluado
  [OK] ΔF1 ≤ 0.15 — actual: 0.0077
  [OK] PSI < 0.20 — actual: 0.0225
  [OK] KS p-val medido

  Calibración
  [OK] ECE calibrado: 0.0381 (S10: 0.033)

  ONNX
  [OK] lstm_s10v2.onnx exportado (opset 17)
  [OK] Latencia < 200 ms en CPU

  MLOps
  [OK] logs/runs.csv actualizado con S10v2

  Pendiente S11
  [PENDIENTE] Frontend